In [ ]:
import io, sys, itertools
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import softmax

rng = np.random.default_rng(12345)
buf = io.StringIO()
def out(*a):
    s = " ".join(str(x) for x in a)
    print(s); buf.write(s + "\n")
def head(t): out("\n" + "=" * 78 + "\n" + t + "\n" + "=" * 78)


head("1. Softmax Jacobian J = diag(p) - p p^T")
def J(p): return np.diag(p) - np.outer(p, p)
best, bestp = 0.0, None
for alpha in [0.05, 0.2, 0.5, 1.0, 3.0]:
    for _ in range(4000):
        n = rng.integers(2, 12)
        p = rng.dirichlet(np.full(n, alpha))
        v = np.linalg.norm(J(p), 2)
        if v > best: best, bestp = v, p
out(f"max spectral norm found by random search over the simplex: {best:.6f}  (at p = {np.round(np.sort(bestp)[::-1][:3], 3)})")
res = minimize(lambda z: -np.linalg.norm(J(softmax(z)), 2), rng.normal(size=6), method="Nelder-Mead", options=dict(xatol=1e-10, fatol=1e-12, maxiter=20000))
out(f"optimised (Nelder-Mead on logits): {-res.fun:.8f}")
p2 = np.array([.5, .5]); eig = np.linalg.eigvalsh(J(p2))
out(f"p=(1/2,1/2): eigenvalues of J = {np.round(eig, 6)}  -> spectral norm 1/2 is attained")
dmax = max(np.max(np.diag(J(rng.dirichlet(np.full(5, a))))) for a in [.1, 1, 5] for _ in range(1000))
out(f"largest DIAGONAL entry p_i(1-p_i) seen: {dmax:.4f} (<= 1/4).  The old text used this as an eigenvalue bound: wrong.")
r2 = max(np.linalg.norm(softmax(a) - softmax(b)) / np.linalg.norm(a - b) for a, b in
         ((rng.normal(size=8) * s, rng.normal(size=8) * s) for s in [.1, 1, 5, 20] for _ in range(5000)))
out(f"max ||softmax(a)-softmax(b)||_2 / ||a-b||_2 over 20000 random pairs: {r2:.4f} (bound 1/2)")
a = np.array([1., -1.]) * 30; b = np.zeros(2)
out(f"l_inf -> l_1 constant: ||softmax(a)-softmax(b)||_1/||a-b||_inf, a=(t,-t)/..: take small t -> "
    f"{np.abs(softmax(np.array([1e-3, -1e-3])) - softmax(np.zeros(2))).sum() / 1e-3:.4f} (bound 1, attained as t->0)")
r1 = max(np.abs(softmax(a) - softmax(b)).sum() / np.max(np.abs(a - b)) for a, b in
         ((rng.normal(size=8) * s, rng.normal(size=8) * s) for s in [.1, 1, 5, 20] for _ in range(5000)))
out(f"max over random pairs: {r1:.4f} (<= 1)")

head("2. Unstructured magnitude pruning: nu(s) = ||theta_s||/||theta||")
P = 1000
theta = np.zeros(P); theta[0] = 1.0
out(f"counterexample to the old Lemma: theta = e_1, s = 1/P: nu = {np.linalg.norm(theta * (np.abs(theta) >= np.sort(np.abs(theta))[-1])):.3f} "
    f"but sqrt(s) = {np.sqrt(1 / P):.3f}  -> ||theta_s|| <= sqrt(s)||theta|| is FALSE")
dists = {"Gaussian": lambda n: rng.normal(size=n), "Laplace": lambda n: rng.laplace(size=n), "Student-t(2)": lambda n: rng.standard_t(2, size=n),
         "lognormal-signed": lambda n: rng.choice([-1, 1], n) * rng.lognormal(0, 1, n)}
rows = []
for name, f in dists.items():
    th = f(100000); a = np.sort(np.abs(th))[::-1]; tot = (a ** 2).sum()
    for s in [0.83, 0.5, 0.25, 0.1]:
        p = int(np.floor(s * len(th))); nu = np.sqrt((a[:p] ** 2).sum() / tot)
        rows.append((name, s, nu, np.sqrt(p / len(th)), nu >= np.sqrt(p / len(th)) - 1e-12, nu <= np.sqrt(s) + 1e-12))
df = pd.DataFrame(rows, columns=["dist", "s", "nu(s)", "sqrt(s)", "nu>=sqrt(floor(sP)/P)", "old claim nu<=sqrt(s) holds"])
out(df.round(4).to_string(index=False))
out("-> nu(s) >= sqrt(floor(sP)/P) always (provable); the old inequality nu <= sqrt(s) is violated in every non-degenerate case.")
th = rng.normal(size=2000); p = 500; acc = []
for _ in range(4000):
    idx = rng.choice(2000, p, replace=False); acc.append((th[idx] ** 2).sum() / (th ** 2).sum())
out(f"random pruning: Monte-Carlo E[nu^2] = {np.mean(acc):.5f}, exact p/P = {p / 2000:.5f}")
W = np.array([[1.0, 1.0], [1.0, -0.9]]); Wm = np.where(np.abs(W) >= np.sort(np.abs(W).ravel())[1], W, 0)
out(f"magnitude pruning can INCREASE the spectral norm: W=[[1,1],[1,-0.9]] ||W||_2={np.linalg.norm(W, 2):.4f}; "
    f"after dropping the smallest entry ||W_s||_2={np.linalg.norm(Wm, 2):.4f}")

head("3. Structured (row) pruning")
I = np.eye(64); s = 0.25; p = 16
out(f"counterexample to the old structured lemma: W = I_64, keep {p} rows: ||W_s||_2/||W||_2 = {np.linalg.norm(I[:p], 2) / np.linalg.norm(I, 2):.3f} "
    f"but sqrt(s) = {np.sqrt(s):.3f}.")
def sample_norms(Wm, p, T=3000):
    m = Wm.shape[0]; v = []
    for _ in range(T):
        idx = rng.choice(m, p, replace=False); v.append(np.linalg.norm(Wm[idx], 2) ** 2)
    return np.array(v)
rows = []
mats = {"Gaussian 256x64": rng.normal(size=(256, 64)), "rank-1 (identical rows)": np.outer(np.ones(256), rng.normal(size=64)),
        "orthonormal rows": np.linalg.qr(rng.normal(size=(64, 256)).T)[0].T, "low-rank (r=4)+noise": rng.normal(size=(256, 4)) @ rng.normal(size=(4, 64)) + 0.1 * rng.normal(size=(256, 64))}
for name, Wm in mats.items():
    m = Wm.shape[0]
    for sf in [0.75, 0.5, 0.25]:
        p = int(sf * m); v = sample_norms(Wm, p, 400); full = np.linalg.norm(Wm, 2) ** 2
        rows.append((name, sf, v.mean() / full, sf, (v <= full + 1e-9).all(), np.sqrt(v.mean() / full)))
df = pd.DataFrame(rows, columns=["matrix", "s", "E[sigma^2(W_s)]/sigma^2(W)", "s (claimed upper bound? no: proven LOWER bound)", "all subsets <= ||W||_2", "rms ratio"])
out(df.round(4).to_string(index=False))
ok = (df.iloc[:, 2] >= df.iloc[:, 3] - 0.01).all()
out(f"-> E[sigma_max^2(W_s)] >= s*sigma_max^2(W) in every case ({ok}); deterministic ||W_s||_2 <= ||W||_2 always holds. "
    "So sqrt(s) is a LOWER limit for the rms contraction, not an upper bound.")

head("4. Single-head Lipschitz bound and head-removal error bound  (norm: max over tokens of the l2 norm)")
def head_fn(X, WQ, WK, WV, WO):
    dk = WQ.shape[1]
    A = X @ WQ @ WK.T @ X.T / np.sqrt(dk)
    return softmax(A, axis=1) @ X @ WV @ WO
def n2inf(Z): return np.max(np.linalg.norm(Z, axis=1))
def make(d, dk, dv, aq, ak, av, ao):
    return [rng.normal(size=(d, dk)) * aq, rng.normal(size=(d, dk)) * ak, rng.normal(size=(d, dv)) * av, rng.normal(size=(dv, d)) * ao]
def bounds(W, B):
    WQ, WK, WV, WO = W; dk = WQ.shape[1]
    bq, bk, bv, bo = [np.linalg.norm(w, 2) for w in W]
    mu = bq * bk / np.sqrt(dk)
    return dict(mu=mu, V=bv * bo, new=bv * bo * (1 + 2 * mu * B ** 2), old=bv * bo * (mu * B / 4 + 1))
def proj(Z, B):
    nr = np.linalg.norm(Z, axis=1, keepdims=True); return Z * np.minimum(1, B / np.maximum(nr, 1e-12))
def search(W, B, T=3, d=3, restarts=60):
    best = 0.0
    def neg(z):
        X = proj(z[:T * d].reshape(T, d), B); X2 = proj(z[T * d:].reshape(T, d), B)
        den = n2inf(X - X2)
        return 0.0 if den < 1e-9 else -n2inf(head_fn(X, *W) - head_fn(X2, *W)) / den
    for _ in range(restarts):
        z0 = rng.normal(size=2 * T * d) * B
        r = minimize(neg, z0, method="Powell", options=dict(maxiter=1200, xtol=1e-5, ftol=1e-7))
        best = max(best, -r.fun)
    return best
rows = []
configs = [(3, 3, 3, 1, 1, 1, 1, 1.0), (3, 3, 3, 1, 1, 1, 1, 3.0), (3, 3, 3, 2, 2, 1, 1, 3.0), (3, 3, 3, 3, 3, 1, 1, 3.0), (3, 3, 3, 3, 3, 1, 1, 6.0), (4, 4, 4, 2, 2, 1, 1, 5.0)]
for (d, dk, dv, aq, ak, av, ao, B) in configs:
    W = make(d, dk, dv, aq, ak, av, ao); b = bounds(W, B)
    found = search(W, B, T=3, d=d, restarts=10)
    rows.append((B, round(b["mu"], 2), round(b["V"], 2), round(found, 3), round(b["new"], 2), round(b["old"], 2), found <= b["new"] + 1e-6, found > b["old"] + 1e-6))
df = pd.DataFrame(rows, columns=["B", "mu=bq*bk/sqrt(dk)", "||V||~bv*bo", "largest ratio found", "NEW bound", "OLD bound", "found<=NEW", "found>OLD (old bound violated)"])
out(df.to_string(index=False))
out(f"-> new bound never violated: {bool(df['found<=NEW'].all())}; configurations where the OLD bound is violated: {int(df['found>OLD (old bound violated)'].sum())} of {len(df)}")

d, dk, dv, H, T, B = 16, 8, 8, 6, 10, 2.0
viol, ratios = 0, []
for trial in range(1000):
    heads = [make(d, dk, dv, rng.uniform(.3, 1.5), rng.uniform(.3, 1.5), rng.uniform(.3, 1.5), rng.uniform(.3, 1.5)) for _ in range(H)]
    X = proj(rng.normal(size=(T, d)) * B, B)
    outs = [head_fn(X, *w) for w in heads]
    for h in range(H):
        mag = n2inf(outs[h]); bnd = B * np.linalg.norm(heads[h][2] @ heads[h][3], 2)
        if mag > bnd + 1e-9: viol += 1
        ratios.append(mag / bnd)
    R = rng.choice(H, size=rng.integers(1, H), replace=False)
    err = n2inf(sum(outs[h] for h in R)); bnd = B * sum(np.linalg.norm(heads[h][2] @ heads[h][3], 2) for h in R)
    if err > bnd + 1e-9: viol += 1
out(f"head output magnitude ||Attn_h(X)||_(2,inf) <= B ||W_V W_O||_2 and removal-error bound: violations in 1000 trials = {viol}; "
    f"largest observed fraction of the bound = {max(ratios):.3f}")

head("5. Retained 'capacity' C_op of the kept heads relative to dense (from the P1/P2/P3 result files)")
try:
    D = pd.concat([pd.read_csv(f"/mnt/user-data/uploads/{f}") for f in ["P1_extra_all_results.csv", "P2_all-results.csv", "P3_all-results.csv"]])
    dn = D[D.method == "dense"].groupby(["model", "task", "seed"]).C_op.first().rename("dense")
    X = D[D.method != "dense"].merge(dn.reset_index(), on=["model", "task", "seed"])
    X["ret"] = X.C_op / X.dense; X["s"] = X.k / 12
    t = X[X.method.isin(["asicr_uniform", "random_uniform", "asicr_inv_uniform"])].groupby(["method", "k"]).agg(s=("s", "first"), retained=("ret", "mean"), min_retained=("ret", "min")).reset_index()
    out(t.round(3).to_string(index=False))
    a = X[X.method == "asicr_uniform"]
    out(f"largest-first (A-SICR) retained fraction >= s in all {len(a)} rows: {bool((a.ret >= a.s - 1e-9).all())};  "
        f"smallest-first (inverse) retained fraction <= s in all rows: {bool((X[X.method=='asicr_inv_uniform'].ret <= X[X.method=='asicr_inv_uniform'].s + 1e-9).all())}")
except Exception as e:
    out("skipped real-data check:", type(e).__name__, e)

open("theory_checks_report.txt", "w").write(buf.getvalue())
print("\nwritten theory_checks_report.txt")
